# Training: Accessing Research-Quality Sea Level Data via the SLSMF API

**Authors:** Nikos Kalligeris, Marinos Charalampakis, Stijn Vermaere, Dias Bakeev, Bart Vanhoorne, Tjess Hernandez

## Overview

This notebook is a hands-on guide for retrieving and working with sea level data through the **IOC Sea Level Station Monitoring Facility (SLSMF) API v2** ([API docs](https://api.ioc-sealevelmonitoring.org/v2/doc)).

By the end of this session, you will be able to:

1. **Configure** your Python environment for API interaction.
2. **Fetch real-time** sea level observations from any IOC station.
3. **Retrieve research-quality** data with fine-grained Quality Control (QC) filtering.
4. **Download tidal harmonic** constituents for a station.
5. **Generate tidal predictions** and compare them against observed data to isolate residuals (e.g., storm surges, tsunamis).

> **Tip:** Save this notebook locally so you can modify the input parameters and re-run the 

## 1. Configuration of the notebook

Before we interact with the API, we need to install and import the required Python packages. This cell handles:

- **Data handling:** `pandas`, `numpy`
- **Visualization:** `matplotlib`, `seaborn`, `plotly`
- **API communication:** `requests`
- **Environment variables:** `python-dotenv` (loads your API key from a `.env` file)
- **Interactive widgets:** `ipywidgets` (for user-friendly input controls)

Run the two cells below to install dependencies and import all modules.

In [2]:
import sys
print(sys.executable)

!{sys.executable} -m pip install requests pandas numpy matplotlib seaborn python-dotenv plotly "nbformat>=4.2.0"

/opt/conda/bin/python


In [4]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from datetime import datetime
from datetime import date
import time
from io import StringIO
import ipywidgets as widgets
from ipywidgets import Box
from IPython.display import JSON

from dotenv import load_dotenv
from urllib.parse import unquote
import matplotlib.pyplot as plt
import copy
from datetime import timedelta

load_dotenv('./env', override=True)

BASE_PATH = "./"
DATAFILES_PATH = os.path.join(BASE_PATH, "datafiles")

## 2. Realtime data

## 2. Real-Time Data

Real-time data provides the most recent sea level observations transmitted by a station. This is useful for operational monitoring, event detection, or quick exploratory analysis.

### 2.1 Input Configuration

To access the API, you need a valid **API Key**. Enter it in the widget below.

> **How to get an API Key:** Follow the instructions in the [API_description.md](https://github.com/SLSMF/API-Documentation/blob/main/API_description.md).


In [5]:
APIKEY = widgets.Text(
    value=os.getenv('API_KEY'),
    placeholder='Type something',
    description='APIKEY:',
    disabled=False   
)
APIKEY

Text(value='bd423717a30e8414189510b75eb36655049d3e1341d86b9b99e83eec2210534ea6f5bd4ecf37f447ec41588f12c1cb871b…

In [6]:
station = widgets.Text(
    value='adak',
    placeholder='IOC code',
    description='IOC:',
    disabled=False   
)
sensor = widgets.Text(
    value='wls',
    description='sensor type:',
    disabled=False   
)
timestart=widgets.DatePicker(
    description='timestart',
    disabled=False,
    value=datetime.strptime('01/01/2025', '%m/%d/%Y')
)
timestop=widgets.DatePicker(
    description='timestop',
    disabled=False,
    value=datetime.strptime('02/01/2025', '%m/%d/%Y')
)

items = [station, sensor, timestart, timestop]
box = Box(children=items)

box

Box(children=(Text(value='adak', description='IOC:', placeholder='IOC code'), Text(value='wls', description='s…

### 2.2 Station Metadata

Before downloading data, it's good practice to inspect the **station metadata**. This tells you:

- Whether the station is **operational**
- Its **geographic coordinates** (latitude, longitude)
- Which **sensors** are available and their last reported values
- The **sampling rate** and **transmission method**

The function below queries the `/stations/{code}` endpoint and returns a structured metadata object.



In [ ]:
def fetch_station_metadata(station = 'adak'):
    url = "https://api.ioc-sealevelmonitoring.org/v2/stations/"+ station
    querystring = {}
    headers = {"X-API-KEY": APIKEY.value, "accept": "aplication/json"}
    response = requests.get(url, headers=headers, params=querystring)
    station_metadata = json.loads(response.text)[0]
    print(response.url)
    from IPython.display import JSON
    station_metadata = JSON(station_metadata, expanded=False)
    return station_metadata

In [ ]:
station_metadata = fetch_station_metadata(station.value)
JSON(station_metadata.data, expanded=False)

### 2.3 Download

The `fetch_realtime_data` function downloads sea level observations from the `/stations/{code}/data` endpoint. It handles:

- **Pagination:** The API limits responses to ~30-day windows, so the function loops through the requested date range in monthly chunks.
- **Unit conversion:** Sea level values are converted from meters to millimeters for consistency.
- **Output:** Data is saved to a local file and loaded into a `pandas` DataFrame indexed by timestamp.



In [ ]:
def fetch_realtime_data(
    station = 'adak',
    timestart = '2025-01-01',
    timestop = '2025-01-10',
    nofilter= 'false',
    allsensors= 'false',
    media_type='application/json',
    skip_gaps_until=None,
    includesensors=['wls'],
):
    timestart = pd.Timestamp(timestart).date()
    timestop = pd.Timestamp(timestop).date()
    loop_start = copy.deepcopy(timestart)
    output_file = DATAFILES_PATH + '/' + station + '.data' 
    if os.path.exists(output_file):
        os.remove(output_file)
        
    while (timestop - loop_start).total_seconds() > 0:
        timestop_parameter = loop_start + timedelta(days=30) 
        timestop_parameter = timestop if timestop_parameter > timestop else timestop_parameter
        url = "https://api.ioc-sealevelmonitoring.org/v2/stations/"+ station + "/data"
        querystring = {"timestart": loop_start.strftime("%Y-%m-%d"), "timestop": timestop_parameter, "includesensors[]": includesensors }
        headers = {"X-API-KEY": APIKEY.value, "accept": "application/json"}
        response = requests.get(url, headers=headers, params=querystring)
        print(response.url)
        if response.status_code != 200:
            raise Exception(f"API Error {response.status_code}: {response.text}")
    
        file = DATAFILES_PATH + '/' + station + '.data'
        with open(file, "a") as f:
            for entry in json.loads(response.text):
                slevel = float(entry['slevel'])*1000
                stime = entry['stime']
                f.write(f'{station} {stime} {slevel} 1\n')
        loop_start = timestop_parameter
    print("\n Downloading realtime data complete, data stored at " + file)

    #read in realtime
    names = ["station", "date", "time", "value", "flag"]
    
    obs = pd.read_csv(
        DATAFILES_PATH + '/' + station + '.data',
        names=names,
        skipinitialspace=True,
        sep=' ',    
        na_values="9.999",
    )
    
    obs["anomaly"] = (obs["value"])
    index = pd.to_datetime(obs["date"] + ' ' + obs['time'])
    obs.index = index
    obs.dropna(subset=["value", "anomaly"], inplace=True)
    
    return obs

In [ ]:
obs = fetch_realtime_data(
    station = station.value,
    timestart = timestart.value.strftime("%Y-%m-%d"),
    timestop = timestop.value.strftime("%Y-%m-%d"),
    includesensors = sensor.value
)
print(obs)

## 3. Research data

### 3.1 Input

Enter the station ID information in the code below. 
-  For the *station_ID* option below, the user can either use the IOC station code, or the SSC ID from the SLSMF catalogue: https://www.ioc-sealevelmonitoring.org/ssc/ . Using the SSC code will allow to identify the preferred sensor for the user-defined dates, for the seal level stations for which multiple sensors are attached to a single SSC code. 
-  For the *sensor* option below, the user can choose whether to harvest data from one ('one-sensor' option) or multiple ('alternate-sensor') sensors. In the case 'one-sensor' is chosen, the API will return data from only one sensor type for the entire period. The selected sensor will be the one that was preferred on the most days during the requested period, filtered by the included sensors parameter. If 'alternate-sensor' is chosen, the API will return data from multiple sensors. The preferred sensor **for each individual day** will be returned, filtered by the included sensors parameter. If no preferred sensor is available on a given day, no data will be returned for that day.
-  For the *includesensors* option below, the user can specify one or more sensors to harvest sea level data from. In combination with the above options, if 'one-sensor' is used and a single sensor is specified in the *includesensors*, the procedure for identifying a preferred sensor for the dates specified is not followed. If more than one sensors are specified in *includesensors*, then the procedure for identifying the preferred sensor will be followed as explained in the above bullet, but restricted to the sensors specified in the *includesensors* field. If *includesensors* is left blank (empty list as []), the code will use all available sensors for the user-defined SSC-ID.

The available *includesensor* options and their interpretation is expained in the *API_description.md* document. To check which sensor(s) are available for a station or site, the users can go to the Catalogue tab on the SLSMF page (https://www.ioc-sealevelmonitoring.org/ssc/), find the SSC-ID on the left column, and then click on the 'detail' button on the right column. The available sensors will appear in the right column of the table in the 'Linked codes' section.

In [ ]:
station = widgets.Text(
    value='adak',
    placeholder='IOC code',
    description='IOC:',
    disabled=False   
)
sensor = widgets.Dropdown(
    options=['one-sensor', 'alternate-sensor'],
    description='sensor',
    disabled=False,
    value='one-sensor'
)
start_date=widgets.DatePicker(
    description='start_date',
    disabled=False,
    value=datetime.strptime('01/01/2025', '%m/%d/%Y')
)

end_date=widgets.DatePicker(
    description='end_date',
    disabled=False,
    value=datetime.strptime('02/01/2025', '%m/%d/%Y')
)

days_per_page = widgets.BoundedIntText(
    min=1,
    max=100,
    value=15,
    description='days per page',
    disabled=False   
)
includesensors = widgets.SelectMultiple(
    options=['rad', 'prs', 'prt', 'prte', 'wls'],
    value=['wls'],
    #rows=10,
    description='includesensors',
    disabled=False
)

display(station, includesensors, sensor, start_date, end_date, days_per_page)

In the following fields, enter the starting and ending day for requesting research-quality sea level data in the format YYYY-MM-DD.
- The *start_date* marks the beginning of the time period for which data is being requested, marking the initial date of the desired data range (this date is included in the results).
- The *end_date* is the last date of the time period for which data is being requested, marking the final date in the specified data range (this date is not included in the results). When the Timestop equals the current date, real time data will be added for the current day. The data will be from the sensor which was selected the most as preferred during the whole period, taking the include sensors field into account.

The *days_per_page* field allows the user to restrict the number of days spanning a page in order to reduce the file size per request, with a maximum of 365 days. 
-  In case *days_per_page* is higher or equal to the days of sea level data requested, a single request will be made to the API server and all data will be downloaded in one page.
-  Otherwise, the sea level data will be requested in chunks of data restricted to a length equal to *days_per_page* - note that if data gaps exist for a station, days with no data do not count in the *days_per_page*, and therefore it may exceed the time period defined by *days_per_page* in terms of dates.

The user inputs below control the relative sea level mean, the output time vector and format of output file. 
- *subtract_30d_average*: when set to *true*, the requested data will be recalculated in reference to the mean sea level of the last ~30 days.
- *flag_qc*: when set to *true*, includes qc flags as extra fields of booleans (T or F).
- *fit_to_sample_rate*: set to *false* to obtain the data at the specific rate or frequency defined by the station. This means that
the data will be provided according to the station's predefined sampling rates. Setting it to *true* organizes the data into predefined time slots based on the transmission rate, essentially normalizing the data to ensure it aligns with the established intervals. Missing but expected records are added with columns slevel = NA and missing = T. This process adjusts the data to fit consistent, standardized time periods, ensuring that it is uniformly distributed according to the rate at which it was transmitted or recorded.
- *media_type*: choose the format in which you would like the requested data to be delivered. Two choices are available: receive the data in text (CSV) format, where values are separated by commas, or in JSON format, which is a structured data format commonly used for representing information in key-value pairs. If *media_type* is left empty, default is 'text/csv'.

In [ ]:
subtract_30d_average = widgets.Dropdown(
    options=['false', 'true'],
    description='subtract_30d_average',
    disabled=False,
    value='false'
)
fit_to_sample_rate = widgets.Dropdown(
    options=['false', 'true'],
    description='fit_to_sample_rate',
    disabled=False,
    value='false'
)
flag_qc = widgets.Dropdown(
    options=['false', 'true'],
    description='flag_qc',
    disabled=False,
    value='true'
)
media_type = widgets.Dropdown(
    options=['application/json', 'text/csv'],
    description='media_type',
    disabled=False,
    value='application/json'
)

display(subtract_30d_average, fit_to_sample_rate, flag_qc, media_type)

The user inputs below, provided in the form of true/false flags, control the individual quality control tools employed by the new SLSMF API. The user needs to set all tool flags to be employed in the post-processing to *true*, and all the others to *false*. Filters never remove entire records. They only remove the sea level value by setting the slevel column to NA. Set *flag_qc* to *true* to see based on which qc parameter this happened. The corresponding qc flag will have as value T.
- *filter_out_of_range*: when set to *true*, this filter sets slevel to NA for points that are significantly higher or lower than the majority of the values within a specified time period. It is designed to identify and remove outliers or anomalies, ensuring that the remaining data more accurately represents typical trends and patterns for that particular time frame.
- *filter_exceeded_neighbours*: when set to *true*, this filter works by comparing the difference between adjacent sea level data points. If the difference between a specific data point and its neighboring values exceeds a defined threshold, the slevel is set to NA for that data point. This helps eliminate abrupt, unusual fluctuations that may not align with the general trend of the surrounding data. Caution should be exercised when using this filter for tsunami events, as the initial tsunami signal may be incorrectly identified as an outlier due to exceeding the neighboring data points.
- *filter_spikes_via_median*: when set to *true*, this filter sets slevel to NA for data points that deviate substantially from a spline-fit curve, which is a smooth, flexible curve that models the underlying trend of the data. By identifying and removing sea levels that significantly differ from this curve, the filter helps to retain only those sea levels that are consistent with the overall trend, improving the accuracy and reliability of the dataset.
- *filter_flat_line*: when set to *true*, this filter addresses data gaps that appear as flat, unchanging segments in the data diagrams, typically indicating periods where no data was recorded or the data was unavailable. By removing these flat-line sections, the filter helps to clean the dataset, ensuring that only continuous, meaningful data is retained for analysis and that gaps in the data do not distort the overall trends or patterns.
- *filter_completeness*: when set to *true*, this filter sets slevel to NA for days with low completeness. A day is flagged when its sum of records is less than 30% of the expected amount, based on specific rate or frequency defined by the station.
- *filter_distinctness*: when set to *true*, this filter sets slevel to NA for days with low distinctness. A day is flagged when it contains less than 10% distinct sea levels, where 100% would be in case every sea level value is unique.
- *filter_shift*: when set to *true*, this filter sets slevel to NA for days with a vertical shift. A day is flagged when the median sea level of the day is outside the 0.1 and 0.9 quantile range of the sea levels in the previous month or until last shift.

A guiding document describing the quality control steps employed by the new SLSMF API service can be found in the *research_data* folder.

In [ ]:
filter_out_of_range = widgets.Dropdown(
    options=['false', 'true'],
    description='out_of_range',
    disabled=False,
    value='false'
)
filter_exceeded_neighbours = widgets.Dropdown(
    options=['false', 'true'],
    description='exceeded_neighbours',
    disabled=False,
    value='false'
)
filter_spikes_via_median = widgets.Dropdown(
    options=['false', 'true'],
    description='spikes_via_median',
    disabled=False,
    value='false'
)
filter_flat_line = widgets.Dropdown(
    options=['false', 'true'],
    description='flat_line',
    disabled=False,
    value='false'
)
filter_completeness = widgets.Dropdown(
    options=['false', 'true'],
    description='completeness',
    disabled=False,
    value='false'
)
filter_distinctness = widgets.Dropdown(
    options=['false', 'true'],
    description='distinctness',
    disabled=False,
    value='false'
)
filter_shift = widgets.Dropdown(
    options=['false', 'true'],
    description='shift',
    disabled=False,
    value='false'
)

from ipywidgets import Box

display(filter_out_of_range, filter_exceeded_neighbours, filter_spikes_via_median, filter_flat_line, filter_completeness, filter_distinctness, filter_shift)

### 3.2 Download

The `fetch_research_data` function queries the `/research/stations/{code}/sensors/{mode}/data` endpoint. It handles pagination, supports both JSON and CSV responses, and returns a consolidated `pandas` DataFrame.


In [ ]:
def fetch_research_data(
    station = 'adak',
    sensor = 'one-sensor',
    includesensors=['wls'],
    timestart = '2025-01-01',
    timestop = '2025-01-10',
    days_per_page = 15,
    subtract_30d_average = 'false',
    fit_to_sample_rate = 'false',
    flag_qc = 'true',
    filter_out_of_range = 'false',
    filter_exceeded_neighbours = 'false',
    filter_spikes_via_median = 'false',
    filter_flat_line = 'false',
    filter_completeness = 'false',
    filter_distinctness = 'false',
    filter_shift = 'false',
    media_type = 'application/json'
):
    """
    Download and read research-quality sea level data from SLSMF API
    Returns a pandas DataFrame with the sea level data
    """
    
    api_url = 'https://api.ioc-sealevelmonitoring.org/v2/research/stations/'
    
    # Calculate number of days and pages
    no_of_days = (pd.Timestamp(timestop).date() - pd.Timestamp(timestart).date()).days
    no_of_pages = int(np.ceil(no_of_days / days_per_page))
    
    # Initialize empty list to store dataframes
    dataframes = []
    
    print(f"Downloading qc data with {no_of_pages} pages in {media_type} format...")
    
    # Loop to download data
    for i in range(1, no_of_pages + 1):
        page = i  # current page number requested
      
        # Building the custom URL
        url = f"{api_url}{station}/sensors/{sensor}/data"
        params = {
            'days_per_page': days_per_page,
            'page': page,
            'timestart': timestart,
            'timestop': timestop,
            'subtract_30d_average': subtract_30d_average,
            'fit_to_sample_rate': fit_to_sample_rate,
            'flag_qc': flag_qc,
            'filter_out_of_range': filter_out_of_range,
            'filter_exceeded_neighbours': filter_exceeded_neighbours,
            'filter_spikes_via_median': filter_spikes_via_median,
            'filter_flat_line': filter_flat_line,
            'filter_completeness': filter_completeness,
            'filter_distinctness': filter_distinctness,
            'filter_shift': filter_shift
        }

        if includesensors and None not in includesensors:
            params['includesensors[]'] = list(includesensors)
      
        # Downloading the data
        print(f'Downloading page {page} / {no_of_pages} of the qc data requested in {media_type} format')
        headers = {
            'X-Api-Key': APIKEY.value,
            'Accept': media_type
        }
        
        response = requests.get(url, params=params, headers=headers)
        print(response.url)
        
        if response.status_code != 200:
            raise Exception(f"API Error {response.status_code}: {response.text}")
            
        response_text = response.text
      
        if media_type == 'application/json':
            data = response.json()
            if 'data' in data:
                tmp = pd.DataFrame(data['data'])
                # Convert stime to datetime
                tmp['stime'] = pd.to_datetime(tmp['stime'], utc=True)
            else:
                tmp = pd.DataFrame()
        else:
            # CSV handling
            tmp = pd.read_csv(StringIO(response_text), skiprows=2)

        if not tmp.empty:
            dataframes.append(tmp)
      
        # Rate limiting (1 second delay)
        time.sleep(1)
    
    # Combine all dataframes
    if not dataframes:
        raise Exception("No sea level data retrieved.")
    
    S = pd.concat(dataframes, ignore_index=True)
    
    S['slevel'] = pd.to_numeric(S['slevel'], errors='coerce')
    
    return S

In [ ]:
sealevel_with_flags = fetch_research_data(
    station = station.value,
    sensor = sensor.value,
    includesensors = includesensors.value,  
    days_per_page = days_per_page.value,
    timestart = pd.Timestamp(start_date.value).date(),
    timestop = pd.Timestamp(end_date.value).date(),
    subtract_30d_average = subtract_30d_average.value,
    fit_to_sample_rate = fit_to_sample_rate.value,
    flag_qc = flag_qc.value,
    filter_out_of_range = filter_out_of_range.value,
    filter_exceeded_neighbours = filter_exceeded_neighbours.value,
    filter_spikes_via_median = filter_spikes_via_median.value,
    filter_flat_line = filter_flat_line.value,
    filter_completeness = filter_completeness.value,
    filter_distinctness = filter_distinctness.value,
    filter_shift = filter_shift.value,
    media_type = media_type.value
)
print(sealevel_with_flags)

Uncomment the line below to save the sea level data to a CSV file.

In [ ]:
# Save to CSV
# sealevel_with_flags.to_csv('SLSMF_tg_data_fit_to_sample_rate.csv', index=False)

### 3.3 Visualization
The code below is used in Python to plot the requested sea level data based on the above user inputs. The plot of the downloaded sea level time series should appear below the code.

#### 3.3.1 Raw vs. Filtered Data

The `plot` function renders a simple time series. We call it twice:

1. **First plot:** All data as returned by the API (including flagged values).
2. **Second plot:** Only values where **all QC flags are `F`** — i.e., data that passed every quality check.

Comparing the two plots side-by-side reveals which values were removed and whether the filtering is appropriate for your use case.


In [ ]:
def plot(S):
    S['slevel'] = pd.to_numeric(S['slevel'], errors='coerce')
    S.dropna(inplace=True)
    # Set plot style
    sns.set(style="whitegrid")
    plt.rcParams['figure.figsize'] = (12, 6)
    
    # plot the data
    plt.figure(figsize=(14, 7))
    
    # Plot the S.stime and S.slevel
    plt.plot(S['stime'], S['slevel'], color='black', linewidth=1.5, label='Sea Level')
    
    # Add legend and labels
    plt.title(f'data for SLSMF station ID = {station}', fontsize=14, fontweight='bold')
    plt.xlabel('date', fontsize=12)
    plt.ylabel('WL (m)', fontsize=12)
    
    # Enable grid
    plt.grid(True, alpha=0.3)
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m-%d'))
    plt.gcf().autofmt_xdate()
    plt.legend(loc='best')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot(sealevel_with_flags)
sealevel_qc = sealevel_with_flags[sealevel_with_flags.missing == 'F']
sealevel_qc = sealevel_qc[sealevel_qc.out_of_range == 'F']
sealevel_qc = sealevel_qc[sealevel_qc.exceeded_neighbours == 'F']
sealevel_qc = sealevel_qc[sealevel_qc.spikes_via_median == 'F']
sealevel_qc = sealevel_qc[sealevel_qc.flat_line == 'F']
sealevel_qc = sealevel_qc[sealevel_qc.completeness == 'F']
sealevel_qc = sealevel_qc[sealevel_qc.distinctness == 'F']

plot(sealevel_qc)

#### 3.3.2 Interactive Plot with QC Flags

The `plot_sea_level_with_qc` function creates an interactive **Plotly** chart where:

- The **black line** shows "clean" sea level (values where no QC flag is raised).
- **Colored markers** highlight data points flagged by each QC filter.
- You can **click a flag in the legend** to hide/show both the markers and the underlying values at those points.

This is especially useful for exploring whether a particular filter is too aggressive or too lenient for your data.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from typing import List, Optional
import plotly.io as pio
import nbformat

def plot_sea_level_with_qc(
    df: pd.DataFrame,
    slevel_col: str = 'slevel',
    time_col: str = 'stime',
    qc_columns: List[str] = None,
    export_path: Optional[str] = None,
):
    """
    Plot sea level data with QC flags.
    Clicking a QC flag in the legend hides BOTH the markers AND the sea level values at those points.
    Expects QC flags to be 'Y' (Yes/True) or 'F' (False).
    """
    
    # Default QC columns if not specified
    if qc_columns is None:
        qc_columns = [
            'out_of_range', 'spikes_via_median', 
            'exceeded_neighbours', 'flat_line', 
            'distinctness', 'completeness', 'shift', 'missing'
        ]
    
    # Filter to only existing QC columns
    qc_columns = [col for col in qc_columns if col in df.columns]
    
    # Make a copy to avoid modifying the original dataframe
    df_plot = df.copy()
    
    # Convert time column to datetime if needed
    if not pd.api.types.is_datetime64_any_dtype(df_plot[time_col]):
        df_plot[time_col] = pd.to_datetime(df_plot[time_col])
    
    # Convert 'NA' strings to NaN for slevel
    df_plot[slevel_col] = pd.to_numeric(df_plot[slevel_col], errors='coerce')
    
    # Sort by time
    df_plot = df_plot.sort_values(time_col).reset_index(drop=True)
    
    # ===== CREATE CLEAN BASE LINE =====
    # Replace slevel with NaN wherever ANY QC flag is 'Y'
    df_plot['slevel_clean'] = df_plot[slevel_col].copy()
    any_flag_mask = pd.Series(False, index=df_plot.index)
    for qc_col in qc_columns:
        any_flag_mask = any_flag_mask | (df_plot[qc_col] == 'T')
    df_plot.loc[any_flag_mask, 'slevel_clean'] = np.nan
    
    # Color mapping for QC flags
    colors = {
        'out_of_range': 'red',
        'spikes_via_median': 'orange',
        'exceeded_neighbours': 'purple',
        'flat_line': 'brown',
        'distinctness': 'pink',
        'completeness': 'cyan',
        'shift': 'green',
        'missing': 'gray'
    }
    
    # ===== BUILD FIGURE =====
    fig = go.Figure()
    
    # 1. Clean base line (gaps where QC flags are raised)
    fig.add_trace(go.Scatter(
        x=df_plot[time_col],
        y=df_plot['slevel_clean'],
        mode='lines',
        name='Sea Level (clean)',
        line=dict(color='black', width=1.5),
        hovertemplate='<b>%{x|%Y-%m-%d %H:%M}</b><br>WL: %{y:.3f} m<extra></extra>',
        legendgroup='clean'
    ))
    
    # 2. For each QC flag, add a LINE trace + MARKER trace (linked by legendgroup)
    for qc_col in qc_columns:
        qc_mask = df_plot[qc_col] == 'T'
        
        if qc_mask.any():
            color = colors.get(qc_col, 'blue')
            label = qc_col.replace('_', ' ').title()
            
            # Line trace showing the actual values at flagged points
            fig.add_trace(go.Scatter(
                x=df_plot.loc[qc_mask, time_col],
                y=df_plot.loc[qc_mask, slevel_col],
                mode='markers',
                name=f'{label} (value)',
                line=dict(color=color, width=1.5),
                hovertemplate=f'<b>{qc_col}</b><br>%{{x|%Y-%m-%d %H:%M}}<br>WL: %{{y:.3f}} m<extra></extra>',
                legendgroup=qc_col,
                showlegend=False  # Don't show separately in legend
            ))
            
            # Marker trace for the flag
            fig.add_trace(go.Scatter(
                x=df_plot.loc[qc_mask, time_col],
                y=df_plot.loc[qc_mask, slevel_col],
                mode='markers',
                name=label,
                marker=dict(
                    color=color,
                    size=8,
                    symbol='circle-open',
                    line=dict(width=2, color=color)
                ),
                hovertemplate=f'<b>{qc_col}</b><br>%{{x|%Y-%m-%d %H:%M}}<br>WL: %{{y:.3f}} m<extra></extra>',
                legendgroup=qc_col
            ))
    
    # Determine station name
    station_name = "Unknown"
    if "sensor" in df_plot.columns:
        station_name = df_plot["sensor"].iloc[0]
    elif "_sensor_id" in df_plot.columns:
        station_name = df_plot["_sensor_id"].iloc[0]

    # Layout
    fig.update_layout(
        title=f'Sea Level Data with QC Flags - Station: {station_name}',
        xaxis_title='Date/Time',
        yaxis_title='Water Level (m)',
        hovermode='x unified',
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        template='plotly_white',
        height=600
    )
    
    fig.update_xaxes(
        tickformat='%Y-%m-%d %H:%M',
        tickangle=-45
    )
    
    if export_path:
        html_path = export_path.replace('.png', '.html')
        fig.write_html(html_path)
        print(f"Saved HTML: {html_path}")
        try:
            fig.write_image(export_path)
            print(f"Saved Image: {export_path}")
        except Exception as e:
            print(f"Could not save PNG (install kaleido): {e}")

    pio.renderers.default = "notebook" 
    fig.show(mode="notebook")

In [ ]:
plot_sea_level_with_qc(
    df=sealevel_with_flags,
    slevel_col='slevel',
    time_col='stime',
    qc_columns=[
        'out_of_range', 'spikes_via_median', 
        'exceeded_neighbours', 'flat_line', 
        'distinctness', 'completeness', 'shift', 'missing'
    ]
)

## 4. Tidal Harmonics

Tidal harmonics are the sinusoidal constituents (e.g., M2, S2, K1) that describe the astronomical tide at a station. The API returns pre computed constituents.

### 4.1 Input

| Parameter | Description |
|-----------|-------------|
| **IOC code** | Station identifier (e.g., `adak`) |
| **sensor** | Sensor type (e.g., `wls`) |
| **datestop** | Reference date for the harmonic analysis. The API uses data leading up to this date. |
| **Latitude correction** | When `true`, applies a latitude-dependent correction to the harmonic constituents. Recommended for most applications. |


In [ ]:
code = widgets.Text(
    value='adak',
    placeholder='IOC code',
    description='IOC code:',
    disabled=False   
)
sensor = widgets.Select(
    options=['rad', 'prs', 'prt', 'prte', 'wls', None],
    value='wls',
    description='sensor',
    disabled=False
)
datestop=widgets.DatePicker(
    description='datestop',
    disabled=False,
    value=date.today()
)
lateral_correction = widgets.Dropdown(
    options=['false', 'true'],
    description='Latitude correction',
    disabled=False,
    value='true'
)

display(code,sensor,datestop,lateral_correction)

### 4.2 Download

The `fetch_harmonics` function queries the `/research/stations/{code}/sensors/{sensor}/tidal-harmonics` endpoint and returns a DataFrame of harmonic constituents with their amplitudes and phases.


In [ ]:
def fetch_harmonics(
    code='abas',
    sensor='rad',
    datestop='2026-05-15',
    lateral_correction='true',
):
    params = locals() #use input parameters as get parameters
    
    api_url = 'https://api.ioc-sealevelmonitoring.org/v2/research/stations/'
    url = f"{api_url}{code}/sensors/{sensor}/tidal-harmonics"

    print(f'Downloading harmonics for {code}/{sensor}')
    headers = {
        'X-Api-Key': APIKEY.value,
        'Accept': 'application/json'
    }

    response = requests.get(url, headers=headers, params=params)
    print(response.url)

    if response.status_code != 200:
        raise Exception(f"API Error {response.status_code}: {response.text}")

    data = response.json()
    
    if not data:
        return data
                    
    print('metadata:', {k: v for k, v in data.items() if k != 'data'})
    data = pd.DataFrame(data['data'])
    
    return data

In [ ]:
harmonics = fetch_harmonics(
    code=code.value, 
    sensor=sensor.value, 
    datestop=datestop.value.strftime("%Y-%m-%d"), 
    lateral_correction=lateral_correction.value
)
print(harmonics)

## 5. Predictions
Using the harmonic constituents, the API can generate **tidal predictions** for any date range. By subtracting the predicted tide from the observed sea level, you obtain the **residual** — the non-tidal component that includes storm surges, tsunamis, and other phenomena.


### 5.1 Input

| Parameter | Description |
|-----------|-------------|
| **IOC code** | Station identifier |
| **sensor** | Sensor type |
| **time_interval** | Output resolution in seconds (60–600, step 60). Default: 60s (1-minute intervals). |
| **start / end date** | Prediction period |
| **days_per_page** | Pagination control (same as research data) |
| **subtract_30d_average** | Detrend relative to the preceding 30-day mean |

In [ ]:
code = widgets.Text(
    value='adak',
    placeholder='IOC code',
    description='IOC code:',
    disabled=False   
)
sensor = widgets.Select(
    options=['rad', 'prs', 'prt', 'prte', 'wls', None],
    value='wls',
    description='sensor',
    disabled=False
)

datestart=widgets.DatePicker(
    description='start date',
    disabled=False,
    value=date(2025, 1, 1)
)

datestop=widgets.DatePicker(
    description='start date',
    disabled=False,
    value=date(2025, 2, 1)
)

includesensors = widgets.SelectMultiple(
    options=['rad', 'prs', 'prt', 'prte', 'wls', None],
    value=['wls'],
    #rows=10,
    description='includesensors',
    disabled=False
)

nofilter = widgets.Dropdown(
    options=['false', 'true'],
    description='No filter',
    disabled=False,
    value='false'
)

allsensors = widgets.Dropdown(
    options=['false', 'true'],
    description='All sensors',
    disabled=False,
    value='false'
)
subtract_30d_average = widgets.Dropdown(
    options=['false', 'true'],
    description='subtract_30d_average',
    disabled=False,
    value='false'
)

time_interval = widgets.IntSlider(
    value=60,
    min=60,
    max=600,
    step=60,
    description='time_interval',
    disabled=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

display(code, sensor, time_interval, datestart, datestop, days_per_page, subtract_30d_average)

### 5.2 Download

The `fetch_predictions` function queries the `/research/stations/{code}/sensors/{sensor}/tidal-harmonics/predictions/data` endpoint with automatic pagination.


In [ ]:
def fetch_predictions(
    code='abas',
    sensor='rad',
    time_interval=60,
    datestart='2026-05-15',
    datestop='2026-05-16',
    days_per_page=15,
    subtract_30d_average='false'
):
    api_url = 'https://api.ioc-sealevelmonitoring.org/v2/research/stations/'
    url = f"{api_url}{code}/sensors/{sensor}/tidal-harmonics/predictions/data"
    headers = {
        'X-Api-Key': APIKEY.value,
        'Accept': 'application/json'
    }

    no_of_days = (pd.to_datetime(datestop) - pd.to_datetime(datestart)).days
    no_of_pages = int(np.ceil(no_of_days / days_per_page))
    print(f'Downloading predictions for {code}/{sensor} ({no_of_pages} pages)...')

    dataframes = []
    for page in range(1, no_of_pages + 1):
        params = {
            'time_interval': time_interval,
            'datestart': datestart,
            'datestop': datestop,
            'days_per_page': days_per_page,
            'page': page,
            'subtract_30d_average': subtract_30d_average
        }
        response = requests.get(url, headers=headers, params=params)
        print(f'Page {page}/{no_of_pages}: {unquote(response.url)}')
        if response.status_code != 200:
            raise Exception(f"API Error {response.status_code}: {response.text}")
        data = response.json()
        if not data or 'data' not in data:
            print(f'No data on page {page}, skipping.')
            continue
        print('metadata:', {k: v for k, v in data.items() if k != 'data'})
        tmp = pd.DataFrame(data['data'])
        if not tmp.empty:
            dataframes.append(tmp)
        time.sleep(1)

    if not dataframes:
        raise Exception("No prediction data retrieved.")

    return pd.concat(dataframes, ignore_index=True)
    

In [ ]:
predictions = fetch_predictions(
    code=code.value,
    sensor=sensor.value,
    time_interval=time_interval.value,
    datestart=datestart.value,
    datestop=datestop.value,
    subtract_30d_average=subtract_30d_average.value,
)
print(predictions)

### 5.3 Visualization: Observations vs. Predictions vs. Residuals

The following examples produce a **four-panel comparison**:

1. **Raw observations** — unfiltered real-time data.
2. **QC-filtered data** — research-quality data after QC flags are applied.
3. **Tidal prediction** — the astronomically predicted water level.
4. **Residual** — the difference (QC data − prediction), offset by the mean sea level. Spikes in this panel often correspond to meteorological or seismic events.


#### 5.3.1 Example: Adak, Alaska

A straightforward example using the Adak tide gauge (`adak`, sensor `wls`) for January 2025.


In [ ]:
station = 'adak'
includesensors = ['wls']
timestart = '2025-01-01'
timestop = '2025-02-01'


obs = fetch_realtime_data(
    station = station,
    timestart = timestart,
    timestop = timestop,
    includesensors = includesensors
)
print('sealevel data with qc flags:')
print(obs)

sealevel_qc = fetch_research_data(
    station = station,
    sensor = 'one-sensor',
    includesensors = includesensors,
    timestart = timestart,
    timestop = timestop,
    subtract_30d_average = 'false',
    fit_to_sample_rate = 'false',
    flag_qc = 'false',
    filter_out_of_range = 'true',
    filter_exceeded_neighbours = 'true',
    filter_spikes_via_median = 'true',
    filter_flat_line = 'true',
    filter_completeness = 'true',
    filter_distinctness = 'true',
    filter_shift = 'false',
)
print('sealevel values with qc:')
print(sealevel_qc)

predictions = fetch_predictions(
    code=station,
    sensor=includesensors[0], #needs to be string
    time_interval=60,
    datestart = timestart,
    datestop = timestop,
    subtract_30d_average='false',
)
print('predictions:')
print(predictions)



# Match time with observations
t = obs.index.to_pydatetime()

sealevel_qc_aligned = sealevel_qc.set_index('stime')
sealevel_qc_aligned.index = sealevel_qc_aligned.index.tz_localize(None)
sealevel_qc_aligned = sealevel_qc_aligned.reindex(obs.index)

predictions_aligned = predictions.set_index('stime')
predictions_aligned.index = pd.to_datetime(predictions_aligned.index)
predictions_aligned = predictions_aligned.reindex(obs.index)

residual = sealevel_qc_aligned.slevel - predictions_aligned.prediction
residual = residual + sealevel_qc_aligned.slevel.mean()


#plot
fig, (ax0, ax1, ax2, ax3) = plt.subplots(figsize=(17, 5), nrows=4, sharey=True, sharex=True)
ax0.plot(t, obs.anomaly/1000, label="Observations", color="C0")
ax1.plot(t, sealevel_qc_aligned.slevel, label="QC", color="C3")
ax2.plot(t, predictions_aligned.prediction, label="Prediction", color="C1")
ax3.plot(t, residual, label="Residual (QC - predictions)", color="C2")
fig.legend(ncol=3, loc="upper center");

#### 5.3.2 Example: Tonga Tsunami at Vanuatu

This example uses the Vanuatu station (`vanu`, sensor `aqu`) during January 2022 — the period of the **Hunga Tonga–Hunga Ha'apai volcanic eruption and tsunami**. The residual panel should clearly show the tsunami signal arriving at this station.


In [ ]:
station = 'vanu'
includesensors = ['aqu']
timestart = '2022-01-01'
timestop = '2022-02-01'

station_metadata = fetch_station_metadata(station)
print(json.dumps(station_metadata.data, indent=2, default=str))


obs = fetch_realtime_data(
    station = station,
    timestart = timestart,
    timestop = timestop,
    includesensors = includesensors
)
print(obs)

sealevel_qc = fetch_research_data(
    station = station,
    sensor = 'one-sensor',
    includesensors = includesensors,
    timestart = timestart,
    timestop = timestop,
    subtract_30d_average = 'false',
    fit_to_sample_rate = 'false',
    flag_qc = 'false',
    filter_out_of_range = 'true',
    filter_exceeded_neighbours = 'true',
    filter_spikes_via_median = 'true',
    filter_flat_line = 'true',
    filter_completeness = 'true',
    filter_distinctness = 'true',
    filter_shift = 'false',
)
print(sealevel_qc)

predictions = fetch_predictions(
    code=station,
    sensor=includesensors[0], #needs to be string
    time_interval=60,
    datestart = timestart,
    datestop = timestop,
    subtract_30d_average='false',
)
print(predictions)



# Match time with observations
t = obs.index.to_pydatetime()

sealevel_qc_aligned = sealevel_qc.set_index('stime')
sealevel_qc_aligned.index = sealevel_qc_aligned.index.tz_localize(None)
sealevel_qc_aligned = sealevel_qc_aligned.reindex(obs.index)

predictions_aligned = predictions.set_index('stime')
predictions_aligned.index = pd.to_datetime(predictions_aligned.index)
predictions_aligned = predictions_aligned.reindex(obs.index)

residual = sealevel_qc_aligned.slevel - predictions_aligned.prediction
residual = residual + sealevel_qc_aligned.slevel.mean()


#plot
fig, (ax0, ax1, ax2, ax3) = plt.subplots(figsize=(17, 5), nrows=4, sharey=True, sharex=True)
ax0.plot(t, obs.anomaly/1000, label="Observations", color="C0")
ax1.plot(t, sealevel_qc_aligned.slevel, label="QC", color="C3")
ax2.plot(t, predictions_aligned.prediction, label="Prediction", color="C1")
ax3.plot(t, residual, label="Residual (QC - predictions)", color="C2")
fig.legend(ncol=3, loc="upper center");

# 6. exercise

Go to the [ioc website](https://www.ioc-sealevelmonitoring.org/map.php) and choose a station of your liking.  
The map has an option plot, where you can choose 'all known station', this will add stations that are down.  
When you hover on a station on the map you can see when we received data for the last time, you can use that to set the dates in the api.  
Use one of the example code blocks.